# 01 · Explorar y preparar datos ECG para Chagas

Este notebook recorre el flujo completo:

1. Cargar el CSV de etiquetas de un dataset (ejemplo: **PTB-XL**).
2. Mapear cada ECG a las clases del proyecto (`scripts/scp_class_map.py`).
3. Ver la **distribución de clases**.
4. Generar un **manifiesto balanceado** (`scripts/plan_balanced_set.py`).
5. Renderizar las imágenes por clase (`scripts/wfdb_to_images.py`) para Teachable Machine.

Requisitos: `pip install -r ../requirements.txt` y los datos descargados en `../data/raw/`.

> Consejo: empieza con `PER_CLASS` bajo (p. ej. 50) para validar el flujo rápido.

In [ ]:
import sys
from pathlib import Path

# hace importables los scripts del proyecto
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'scripts'))

# --- Configura estas rutas segun tu dataset ---
DATASET   = 'ptbxl'                                  # 'ptbxl' o 'code15'
LABELS    = ROOT / 'data' / 'raw' / 'ptbxl' / 'ptbxl_database.csv'
RAW_DIR   = ROOT / 'data' / 'raw' / 'ptbxl'
ID_COL    = 'filename_hr'
LABEL_COL = 'scp_codes'   # PTB-XL: columna con dict serializado
BOOL_COLS = []            # CODE-15%: p. ej. ['RBBB', '1dAVb']
OUT_IMAGES = ROOT / 'data' / 'images'
MANIFEST   = ROOT / 'data' / 'manifest.csv'
PER_CLASS  = 50
print('OK — configuracion cargada')

## 1 · Cargar etiquetas y asignar clases

In [ ]:
import pandas as pd
from scp_class_map import map_labels

df = pd.read_csv(LABELS)
print(f'{len(df)} registros en {LABELS.name}')

def row_classes(row):
    if BOOL_COLS:
        active = ' '.join(c for c in BOOL_COLS if str(row.get(c, '')).strip() in ('1', '1.0', 'True', 'true'))
        return map_labels(active)
    return map_labels(str(row.get(LABEL_COL, '')))

df['clases'] = df.apply(row_classes, axis=1)
df[[ID_COL, 'clases']].head(10)

## 2 · Distribucion de clases

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

counts = Counter(c for lst in df['clases'] for c in lst)
counts = dict(sorted(counts.items(), key=lambda kv: kv[1], reverse=True))
print(counts)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(list(counts), list(counts.values()), color='#c0392b')
ax.set_title('Distribucion de clases (' + DATASET + ')')
ax.set_ylabel('n.o de ECG')
for i, v in enumerate(counts.values()):
    ax.text(i, v, str(v), ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

## 3 · Generar manifiesto balanceado

Submuestrea las clases grandes para no sesgar el modelo.

In [ ]:
import subprocess

cmd = [sys.executable, str(ROOT / 'scripts' / 'plan_balanced_set.py'),
       '--labels', str(LABELS), '--id-col', ID_COL,
       '--out', str(MANIFEST), '--per-class', str(PER_CLASS)]
cmd += (['--bool-cols', *BOOL_COLS] if BOOL_COLS else ['--label-col', LABEL_COL])
print(subprocess.run(cmd, capture_output=True, text=True).stdout)

## 4 · Renderizar imagenes por clase

Consume el manifiesto y escribe PNGs en `data/images/<clase>/`. Requiere los datos WFDB
en `RAW_DIR`. Cada subcarpeta se sube como una clase en Teachable Machine.

In [ ]:
cmd = [sys.executable, str(ROOT / 'scripts' / 'wfdb_to_images.py'),
       '--input', str(RAW_DIR), '--format', 'wfdb',
       '--manifest', str(MANIFEST), '--out', str(OUT_IMAGES)]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-2000:])
print(res.stderr[-1000:])

## 5 · Vista previa de un ejemplo por clase

In [ ]:
from PIL import Image

for cls_dir in sorted((OUT_IMAGES).glob('*')):
    if not cls_dir.is_dir():
        continue
    imgs = list(cls_dir.glob('*.png'))
    if not imgs:
        continue
    plt.figure(figsize=(5, 5))
    plt.imshow(Image.open(imgs[0]))
    plt.title(cls_dir.name + f'  (n={len(imgs)})')
    plt.axis('off'); plt.show()

## Siguientes pasos

1. Sube `data/images/` a [Teachable Machine](https://teachablemachine.withgoogle.com/) (Image Project),
   arrastrando cada subcarpeta como una clase.
2. Entrena y exporta el modelo (TensorFlow.js / Keras / TFLite).
3. Revisa el balance: si una clase sigue siendo escasa, sube su `PER_CLASS` o combina datasets.